In [1]:
import pennylane as QuantumIoT
import numpy as np
import matplotlib.pyplot as plt

In [2]:
coeffs = [
    -0.75,
    -1.25,
    -2.0,
     2.75
]

ops = [
    QuantumIoT.Z(0),
    QuantumIoT.Z(1),
    QuantumIoT.Z(2),
    QuantumIoT.Z(0) @ QuantumIoT.Z(1)
]

H = QuantumIoT.Hamiltonian(coeffs, ops)

In [3]:
print("The Hamiltonian is:")
print(H)

The Hamiltonian is:
-0.75 * Z(0) + -1.25 * Z(1) + -2.0 * Z(2) + 2.75 * (Z(0) @ Z(1))


In [4]:
def ansatz(params):

    for i in range(3):
        QuantumIoT.RY(params[i], wires=i)

    QuantumIoT.CNOT(wires=[0,1])
    QuantumIoT.CNOT(wires=[1,2])

    for i in range(3):
        QuantumIoT.RY(params[i+3], wires=i)

In [5]:
dev = QuantumIoT.device("default.qubit", wires=3)

In [6]:
@QuantumIoT.qnode(dev)
def VQESensorEnergy(params):

    ansatz(params)

    return QuantumIoT.expval(H)

In [7]:
params = QuantumIoT.numpy.random.random(6)
opt = QuantumIoT.AdamOptimizer(stepsize=0.1)

In [8]:
for step in range(100):

    params = opt.step(VQESensorEnergy, params)

    if (step + 1) % 10 == 0:
        energy = VQESensorEnergy(params)
        print(f"Step {step+1:3d} | Total Sensor Energy = {energy:.6f}")      

Step  10 | Total Sensor Energy = -2.640142
Step  20 | Total Sensor Energy = -3.981587
Step  30 | Total Sensor Energy = -4.292347
Step  40 | Total Sensor Energy = -4.426061
Step  50 | Total Sensor Energy = -4.768463
Step  60 | Total Sensor Energy = -5.164333
Step  70 | Total Sensor Energy = -5.247209
Step  80 | Total Sensor Energy = -5.236004
Step  90 | Total Sensor Energy = -5.242955
Step 100 | Total Sensor Energy = -5.249189


In [9]:
# ============================================
# Final Results
# ============================================

final_energy = VQESensorEnergy(params)

print("\nOptimized Parameters:")
print(params)

print("\nGround-State Energy:")
print(final_energy)


Optimized Parameters:
[-3.20847402  0.65691253  1.5742781   0.04068864  3.79886217 -1.57782182]

Ground-State Energy:
-5.249189049990134


In [10]:
@QuantumIoT.qnode(dev, shots=2048)
def VQESamplingSensor(params):

    ansatz(params)

    return QuantumIoT.sample()

In [11]:
samples = VQESamplingSensor(params)

In [12]:
print("\n================ SAMPLES FROM VQE SENSOR ENERGY CIRCUIT ================\n")
print(samples)


================ SAMPLES FROM VQE SENSOR ENERGY CIRCUIT ================

[[1 0 0]
 [1 0 0]
 [1 0 0]
 ...
 [1 0 0]
 [1 0 0]
 [1 0 0]]


In [ ]:
most_from_samples = np.array([tuple(sample) for sample in samples])
(unique, counts) = np.unique(most_from_samples, axis=0, return_counts=True)

print("\nUnique bitstrings and their counts:")

for bitstring, count in zip(unique, counts):
    print(f"{bitstring} -> {count} times")


Unique bitstrings and their counts:
[0 1 0] -> 3 times
[1 0 0] -> 2045 times
